In [22]:
from langgraph.graph import StateGraph,START,END
from langchain_ollama import ChatOllama
from typing import TypedDict,Literal
from pydantic import BaseModel,Field

In [23]:
class reviewstate(TypedDict):
    review: str
    sentiment : Literal['postive','Negative']
    dignosis : dict
    response : str

In [24]:
class diagnosis(BaseModel):
    issue_type : str
    tone : str
    urgency : str
class sentimentsschema(BaseModel):
    sentiment : Literal["postive","Negative"]=Field(descriptions="sentiment of review")

/var/folders/q6/qfcg7gvj3_sfpjqdsf6x2k3w0000gn/T/ipykernel_42934/3638380913.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descriptions'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  sentiment : Literal["postive","Negative"]=Field(descriptions="sentiment of review")


In [25]:
model=ChatOllama(model='llama3.2:latest')
structured_diagnosis_llm=model.with_structured_output(diagnosis)
sentiment_llm=model.with_structured_output(sentimentsschema)


In [26]:
prompt=f"if the sentiment is negative so find out what is the negative review the software is too bad i am so done with this software hangs everytime"
structured_diagnosis_llm.invoke(prompt)

diagnosis(issue_type='software issue', tone='negative', urgency='high')

In [37]:
def find_sentiment(state:reviewstate):
    prompt=f"for the following review write a sentiment : {state['review']}"
    state['sentiment']=sentiment_llm.invoke(prompt).sentiment
    return state

def check_sentiment(state:reviewstate)->Literal["run_diagnosis","postitive_response"]:
    
    if state['sentiment']=="postive":
        return "postitive_response"
    if state['sentiment']=="Negative":
        return "run_diagnosis"

def run_diagnosis(state :reviewstate):
    prompt=f"if the sentiment is negative so find out what is the negative review.{state['review']}"
    diagnosis={}
    response=structured_diagnosis_llm.invoke(prompt)

    # diagnosis['issue_type']=structured_diagnosis_llm.invoke(prompt).issue_type
    # diagnosis['tone']=structured_diagnosis_llm.invoke(prompt).tone
    # diagnosis['urgency']=structured_diagnosis_llm.invoke(prompt).urgency
    
    state['dignosis']=response.model_dump()
    return state


def postitive_response(state:reviewstate):
    prompt="The user has given a positive review on {state['review']}.please frame a response for user"
    state['response']=model.invoke(prompt).content
    return state

def negative_response(state:reviewstate):
    prompt="The user has given a negative review on {state['review']}.The dignosis of review is {state['dignosis']} please frame a response for user"
    state['response']=model.invoke(prompt).content
    return state



    

In [38]:
graph=StateGraph(reviewstate)

In [39]:
graph.add_node('find_sentiment',find_sentiment)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('postitive_response',postitive_response)
graph.add_node('negative_response',negative_response)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('postitive_response',END)
graph.add_edge('negative_response',END)

workflow=graph.compile()



In [40]:
initial_state={"review":"the software is too bad i am so done with this software hangs everytime"}

In [42]:
workflow.invoke(initial_state)

{'review': 'the software is too bad i am so done with this software hangs everytime',
 'sentiment': 'Negative',
 'dignosis': {'issue_type': 'Negative Review',
  'tone': 'Angry',
  'urgency': 'High'},
 'response': 'Here\'s a possible response:\n\n"Thank you for sharing your feedback about your recent experience with us. We apologize that our service did not meet your expectations, as indicated by your review and diagnosis.\n\nWe take all complaints seriously and are truly sorry that we fell short of your standards. Can you please tell us more about what specifically didn\'t meet your expectations? Your input will help us identify areas for improvement and make necessary changes to prevent similar issues in the future.\n\nIf there\'s anything we can do to rectify the situation or provide a resolution, please don\'t hesitate to let us know. We value your feedback and appreciate your loyalty."\n\nThis response acknowledges the user\'s concerns, apologizes for any inconvenience, and invites